In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [6]:
def search(query):
    results = index.search(query=query, num_results=5)
    return results

def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the documentation database for relevant results based on a query string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up in the index"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "add_entry",
            "description": "Add a new documentation entry to the index.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The source filename"},
                    "title": {"type": "string", "description": "The title of the entry"},
                    "description": {"type": "string", "description": "A short description"},
                    "content": {"type": "string", "description": "The full content"}
                },
                "required": ["filename", "title", "description", "content"],
                "additionalProperties": False
            }
        }
    }
]


In [4]:
question = "How do I create a dahsbord in Evidently?"

In [5]:
from typing import Literal
from pydantic import BaseModel, Field

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"]
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [8]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self,tool_call):
        """Execute the function based on tool call"""
        try:
            arguments = json.loads(tool_call.function.arguments)
            name = tool_call.function.name
            
            if name == 'search':
                result = search(**arguments) 
            elif name == 'add_entry':
                result = add_entry(**arguments)
            else: 
                result = f'Unknown tool: "{name}"'

            # Truncate to avoid 413
            result_str = json.dumps(result)
            if len(result_str) > 2000:
                result_str = result_str[:2000] + '... [truncated]'    
            
            return {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result_str),
            }
        except Exception as e:
            print(f"Error in make_call: {e}")
            return {
                "role": "tool", 
                "tool_call_id": tool_call.id,
                "content": f"Error: {str(e)}",
            }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=message_history,
                tools=self.tools,
                tool_choice="auto",
                parallel_tool_calls=False,
            )
        
            assistant_message = response.choices[0].message

            clean_message = {"role": assistant_message.role, "content": assistant_message.content}
            if assistant_message.tool_calls:
                clean_message["tool_calls"] = assistant_message.tool_calls
            message_history.append(clean_message)

            has_function_calls = False

            for tool_call in (assistant_message.tool_calls or []):
                print(f'executing {tool_call.function.name}({tool_call.function.arguments})...')
                tool_call_output = self.make_call(tool_call)
                message_history.append(tool_call_output)
                has_function_calls = True

            if not has_function_calls:
                if assistant_message.content:
                    print('ASSISTANT:', assistant_message.content)
                break
                    
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        


In [9]:
class StructuredAgent1(Agent):
    
    def __init__(self, llm_client, model, instructions, tools, output_type):
        super().__init__(llm_client, model, instructions, tools)
        self.output_type = output_type
    
    def structured_loop(self, user_prompt, message_history=None):
        # Step 1: run the normal tool loop (already written!)
        message_history = self.loop(user_prompt, message_history)
        
        # Step 2: make a separate call with response_format, no tools
        schema = self.output_type.model_json_schema()
        message_history.append({
            "role": "user",
            "content": f"Now format your answer as a JSON object matching this schema:\n{schema}\n\nReturn only the JSON object."
        })
        
        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=message_history,
            response_format={"type": "json_object"}
        )
        
        raw = response.choices[0].message.content
        output = self.output_type(**json.loads(raw))
        return message_history, output

In [10]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [13]:
structured_agent = StructuredAgent1(
    llm_client=groq_client,
    model="llama-3.1-8b-instant",
    instructions=instructions,
    tools=tools,
    output_type=RAGResponse
)

messages, output = structured_agent.structured_loop("How do I create a dashboard in Evidently?")
print(output.answer)
print(output.confidence)
print(output.followup_questions)

executing search({"query":"Create a dashboard in Evidently"})...

executing search({"query":"Dashboards Tabs Evidently"})...

ASSISTANT: To answer the user's question, the user wants to know how to create a dashboard in Evidently.

The first search was about "Create a dashboard in Evidently" which yielded some results, including "By default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them."

The second search was about "Dashboards Tabs Evidently" which provided some instructions on creating custom tabs and pre-built tabs in Evidently.

Synthesizing the results, to create a dashboard in Evidently, the user needs to start by adding panels to the dashboard. By default, new panels appear on a single dashboard, but the user can add multiple tabs to organize them. The user can also add custom tabs or use pre-built tabs, which are dashboard templates with preset panel combinations.

However, to populate the panels, the user must first add reports with evalu

In [14]:
print(output.answer)
print()
print(f"Found answer: {output.found_answer}")
print(f"Confidence: {output.confidence}")
print(f"Confidence explanation: {output.confidence_explanation}")
print(f"Answer type: {output.answer_type}")
print(f"Follow-up questions: {output.followup_questions}")

To create a dashboard in Evidently, start by adding panels to the dashboard. By default, new panels appear on a single dashboard, but you can add multiple tabs to organize them. You can add custom tabs or use pre-built tabs, which are dashboard templates with preset panel combinations. However, to populate the panels, you must first add reports with evaluation results to the project.

Here are the steps to create a dashboard in Evidently:
- Add panels to the dashboard.
- Add custom tabs or use pre-built tabs.
- Populate the panels with reports that contain evaluation results.

Found answer: True
Confidence: 0.8
Confidence explanation: The confidence level is moderate because the search results provided clear instructions on creating a dashboard in Evidently, but there may be minor variations or edge cases that are not covered.
Answer type: how-to
Follow-up questions: ['How do I create a custom tab in Evidently?', 'What are the pre-built tabs available in Evidently?', 'How do I add repo

In [15]:
def create_fake_tool(output_type, name="structure_result"):
    schema = output_type.model_json_schema()
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": "Call this when ready to give the final structured answer.",
            "parameters": schema
        }
    }

In [21]:
import re
class StructuredAgent2(Agent):
    
    def __init__(self, llm_client, model, instructions, tools, output_type):
        super().__init__(llm_client, model, instructions, tools)
        self.output_type = output_type
        self.tools.append(create_fake_tool(output_type))
    
    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions + "\n\nIMPORTANT: When you are ready to give your final answer, you MUST call the 'structure_result' tool with your answer. Do not return plain text."},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            try:
                response = self.llm_client.chat.completions.create(
                    model=self.model,
                    messages=message_history,
                    tools=self.tools,
                    tool_choice="auto",
                    parallel_tool_calls=False,
                )
            except Exception as e:
                error_body = e.response.json()
                failed = error_body.get('error', {}).get('failed_generation', '')
                match = re.search(r'<function=\w+>(.*?)</function>', failed, re.DOTALL)
                if match:
                    output = self.output_type(**json.loads(match.group(1)))
                    return message_history, output
                raise
            
            assistant_message = response.choices[0].message

            clean_message = {"role": assistant_message.role, "content": assistant_message.content}
            if assistant_message.tool_calls:
                clean_message["tool_calls"] = assistant_message.tool_calls
            message_history.append(clean_message)

            has_function_calls = False

            for tool_call in (assistant_message.tool_calls or []):
                if tool_call.function.name == "structure_result":
                    # this is the fake tool - parse and break
                    output = self.output_type(**json.loads(tool_call.function.arguments))
                    return message_history, output
                else:
                    # real tool - execute normally
                    tool_call_output = self.make_call(tool_call)
                    message_history.append(tool_call_output)
                    has_function_calls = True

            if not has_function_calls:
                if assistant_message.content:
                    print('ASSISTANT:', assistant_message.content)
                return message_history, None
                    
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        

    

In [22]:
structure_agent_2 = StructuredAgent2(
    llm_client=groq_client,
    model="llama-3.1-8b-instant",
    instructions=instructions,
    tools=tools,
    output_type=RAGResponse
)

_, output = structure_agent_2.loop('bashboards')

ASSISTANT: </function><structure_result>{"answer": "### What are Board Templates?\n\nBoard templates are pre-designed templates for creating boards in the Evidence app. They provide a pre-defined structure and layout to help users set up boards quickly and efficiently.\n\n### How to use Board Templates\n\nTo use board templates, simply navigate to the Boards section and click on the \"New Board\" button. Select the template you want to use from the list of available templates, and then customize the template as needed to suit your specific use case.\n\n### Benefits of using Board Templates\n\nUsing board templates can save time and increase productivity by providing a pre-designed structure for boards. They can also help ensure consistency across multiple boards and make it easier to set up new boards.\n\n### Found Answer": "true", "confidence": "0.8", "confidence_explanation": "Based on the availability of board templates in the Evidence app.", "answer_type": "how-to", "followup_quest